[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/05_llms_finetuning_retrieval/llm_systems_lab.ipynb)

# LLM Systems Lab — PEFT Memory, Decoding, Scaling, Retrieval, Evaluation

**Session 5 armory notebook.** Every claim in Lessons 1–5 that can be checked with arithmetic is checked here. CPU-only, no model downloads, no network — the whole notebook runs in well under a minute.

▶ **[Open this notebook in Colab](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/05_llms_finetuning_retrieval/llm_systems_lab.ipynb)** (text link, in case the badge above does not render in your viewer)

| Part | Lesson | What it verifies |
|---|---|---|
| 1 | 1 — PEFT | The 16-bytes-per-parameter breakdown, the LoRA rank sweep, and why low rank is enough |
| 2 | 2 — Decoding | Top-k's failure mode, measured; and the temperature × top-p interaction |
| 3 | 3 — Scaling | Compute-optimal allocation solved numerically, then re-solved with inference cost |
| 4 | 4 — RAG | BM25 and dense retrieval failing on disjoint queries, and RRF fixing both |
| 5 | 5 — Evaluation | Standard error, McNemar's test, statistical power, and selection bias |

> **What to record:** nothing here belongs in your Metric Vault. Every number in this notebook is computed from published formulas or synthetic data — none of it is evidence about *your* projects. Use it to make the arithmetic automatic, then quote only your own run logs in an interview.

*Dependencies pinned and last verified 2026-07-29.*

In [ ]:
%pip install -q "numpy>=1.26" "matplotlib>=3.8"

In [ ]:
import numpy as np, math
from math import comb
import matplotlib.pyplot as plt

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (10, 4); plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = .3
print("numpy", np.__version__)

---
## Part 1 — Where fine-tuning memory actually goes

Lesson 1's claim: mixed-precision AdamW costs about **16 bytes per parameter**, and only 2 of them are the weights. Everything about PEFT follows from that ratio, so compute it rather than remembering it.

In [ ]:
N = 7e9  # 7B parameters

parts = {"weights fp16": 2, "gradients fp16": 2, "Adam m fp32": 4, "Adam v fp32": 4, "master weights fp32": 4}
total = sum(parts.values())

print("Full fine-tuning, per parameter:")
for k, v in parts.items():
    print(f"  {k:<22} {v:>2} bytes   ->  {v*N/1e9:7.1f} GB at 7B")
print(f"  {'TOTAL':<22} {total:>2} bytes   ->  {total*N/1e9:7.1f} GB at 7B")
print(f"\n  Weights are {parts['weights fp16']}/{total} of the cost.")
print(f"  The other {total - parts['weights fp16']} bytes exist ONLY for trainable parameters.")

In [ ]:
def regime_gb(n_params=7e9, base_bytes=2, trainable_frac=1.0):
    '''Weights + per-trainable-parameter training state. Excludes activations.'''
    return (n_params * base_bytes + n_params * trainable_frac * 14) / 1e9

lora_frac = 0.004   # ~0.4% trainable, typical for r=8 on attention + MLP

for name, gb in [
    ("Full fine-tune (fp16 base)", regime_gb(base_bytes=2,   trainable_frac=1.0)),
    ("LoRA r=8      (fp16 base)",  regime_gb(base_bytes=2,   trainable_frac=lora_frac)),
    ("QLoRA r=8     (NF4 base)",   regime_gb(base_bytes=0.5, trainable_frac=lora_frac)),
]:
    print(f"{name:<30} {gb:>7.1f} GB")

**Read the two steps separately, because interviewers ask which is which.**

- 112 GB → 14.4 GB is **LoRA**: the 14-byte training-state term disappears for the frozen base.
- 14.4 GB → 3.9 GB is **QLoRA**: the frozen base itself drops from 2 bytes/param to 0.5 in NF4.

They are different terms and they compose. Neither touches **activations**, which is why gradient checkpointing appears alongside QLoRA rather than instead of it.

In [ ]:
d = k = 4096
print(f"{'rank':>5} {'params r(d+k)':>15} {'% of dk':>10} {'MB fp16':>9}")
for r in [1, 2, 4, 8, 16, 32, 64, 128]:
    p = r * (d + k)
    print(f"{r:>5} {p:>15,} {100*p/(d*k):>9.2f}% {p*2/1e6:>8.2f}")

### Why is such a low rank enough?

Lesson 1 cites the **low intrinsic dimension** of fine-tuning updates. Simulate one: build an update whose true structure is rank 12, add noise worth 15% of its norm, then ask how much of the update a rank-$r$ approximation captures. (The SVD gives the *best possible* rank-$r$ approximation, so this is an upper bound on what any rank-$r$ adapter could learn.)

In [ ]:
D, rank_true = 512, 12
U = np.linalg.qr(np.random.randn(D, rank_true))[0]
V = np.linalg.qr(np.random.randn(D, rank_true))[0]
S = np.diag(np.linspace(1.0, 0.2, rank_true))

delta = U @ S @ V.T
delta /= np.linalg.norm(delta)
noise = np.random.randn(D, D)
noise *= 0.15 / np.linalg.norm(noise)          # noise worth 15% of the update's norm
delta_noisy = delta + noise

sv = np.linalg.svd(delta_noisy, compute_uv=False)
energy = np.cumsum(sv**2) / np.sum(sv**2)

print(f"{'rank':>5} {'captured energy':>17} {'adapter params vs full':>24}")
for r in [1, 2, 4, 8, 12, 16, 32, 64]:
    print(f"{r:>5} {100*energy[r-1]:>16.2f}% {100*r*2*D/(D*D):>23.1f}%")

**Rank 12 captures ~98% of the update using under 5% of the parameters**, and everything past rank 12 buys almost nothing — the remaining ~2% is the noise floor, which no adapter should want to fit anyway.

That shape is the argument for LoRA: *if* the update you need is close to low-rank, a low-rank parameterization loses very little. It is also the argument for the honest caveat in Lesson 1 — when the required update is genuinely high-rank, as on a large distribution shift, the same curve says you will lose real capacity.

---
## Part 2 — Top-k's failure mode, measured

Lesson 2's claim: a fixed $k$ is wrong because distributions differ in sharpness. Build two synthetic next-token distributions over a 50k vocabulary — a peaked one ("The capital of France is") and a flat one ("She opened the door and") — and measure what each truncation method actually keeps.

In [ ]:
VOCAB = 50000

def zipf_logits(alpha, vocab=VOCAB):
    ranks = np.arange(1, vocab + 1)
    p = 1.0 / ranks**alpha
    return np.log(p / p.sum())

peaked = zipf_logits(2.2)   # confident context
flat   = zipf_logits(0.9)   # open-ended context

def softmax(z, T=1.0):
    z = z / T
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def entropy(p):
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def nucleus_size(p, top_p):
    s = np.sort(p)[::-1]
    return int(np.searchsorted(np.cumsum(s), top_p) + 1)

def mass_kept_topk(p, k):
    return float(np.sort(p)[::-1][:k].sum())

for name, z in [("peaked (confident)", peaked), ("flat (open-ended)", flat)]:
    p = softmax(z)
    print(f"{name:<22} top-1 prob {p.max():.3f}   entropy {entropy(p):5.2f} nats"
          f"   nucleus@p=0.9 {nucleus_size(p, 0.9):>6,}   mass kept by k=50 {mass_kept_topk(p, 50):.3f}")

**One value of $k$, two opposite failures.** In the peaked context, $k=50$ keeps 99.5% of the mass — meaning 46 of those 50 tokens are near-zero-probability junk that sampling can still select. In the flat context, $k=50$ keeps only 27% of the mass, cutting away the majority of legitimate continuations.

Top-p's nucleus moves from **4 tokens to ~25,000** across the same two contexts, because the distribution's entropy sets the cut instead of a hyperparameter.

In [ ]:
temps = [0.5, 0.7, 1.0, 1.2, 1.5]
print(f"{'T':>5} {'nucleus, flat ctx':>20} {'nucleus, peaked ctx':>22}")
rows = []
for T in temps:
    a, b = nucleus_size(softmax(flat, T), 0.9), nucleus_size(softmax(peaked, T), 0.9)
    rows.append((T, a, b))
    print(f"{T:>5} {a:>20,} {b:>22,}")

fig, ax = plt.subplots()
ax.semilogy(temps, [r[1] for r in rows], "o-", label="flat context (open-ended)")
ax.semilogy(temps, [r[2] for r in rows], "s-", label="peaked context (confident)")
ax.set_xlabel("temperature"); ax.set_ylabel("nucleus size at top-p = 0.9 (log scale)")
ax.set_title("The same top-p admits wildly different candidate sets as temperature changes")
ax.legend(); plt.show()

**Temperature and top-p are not independent knobs.** Temperature is applied *first*, so it reshapes the distribution before the nucleus is computed: at the same `top_p=0.9`, the flat context admits ~640 tokens at T=0.7 and ~34,000 at T=1.2.

That is why a configuration copied from another project behaves nothing like it did there, and why **min-p** — a threshold relative to the maximum probability — is more stable at high temperature.

---
## Part 3 — Compute-optimal allocation, solved

Lesson 3's claim: with $C \approx 6ND$ fixed, minimizing loss allocates parameters and tokens in roughly equal proportion. Solve it numerically using the Chinchilla parametric form

$$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}$$

with the coefficients as published: $E = 1.69$, $A = 406.4$, $B = 410.7$, $\alpha = 0.34$, $\beta = 0.28$.

In [ ]:
E, A, B, alpha, beta = 1.69, 406.4, 410.7, 0.34, 0.28

def chinchilla_loss(N_, D_):
    return E + A / N_**alpha + B / D_**beta

def optimal_allocation(C):
    '''Minimize L(N, D) subject to 6ND = C, by scanning N.'''
    Ns = np.logspace(7, 13, 4000)
    Ds = C / (6 * Ns)
    L = chinchilla_loss(Ns, Ds)
    i = int(np.argmin(L))
    return Ns[i], Ds[i], L[i]

print(f"{'budget C (FLOPs)':>18} {'N*':>10} {'D*':>13} {'tokens/param':>14} {'loss':>8}")
for C in [1e21, 1e22, 1e23, 5.76e23, 1e24, 1e25]:
    n, dd, l = optimal_allocation(C)
    print(f"{C:>18.2e} {n/1e9:>8.1f}B {dd/1e9:>11.0f}B {dd/n:>14.1f} {l:>8.3f}")

> **A discrepancy worth knowing — and worth raising in an interview.**
>
> The scan above uses the parametric coefficients exactly as published, and it returns roughly **50–120 tokens per parameter**, not the famous ≈20. That is not a bug in the code: the Chinchilla paper's third approach (the parametric fit) is **inconsistent with its own first two approaches** (the isoFLOP and training-curve analyses), which are where the 20-tokens-per-parameter rule comes from. Besiroglu et al. (2024) replicated the fit and found the published coefficients irreconcilable with the paper's own data; their re-estimated coefficients land much closer to the 1:1 scaling the headline claims.
>
> Two things to take from this. **Quote the 20:1 rule as coming from the isoFLOP analysis**, which is what it rests on. And notice the general lesson: a widely cited number can be inconsistent with a table inside the same paper for two years before anyone checks. That is the habit — recompute the claim — that this whole notebook exists to build.
>
> Besiroglu et al. 2024, *Chinchilla Scaling: A Replication Attempt* (arXiv:2404.10102).

What survives the discrepancy is the **shape**, which is what the interview question is actually about: the optimum is interior (neither all parameters nor all data), the ratio drifts slowly with budget, and the loss approaches a floor $E$ rather than zero.

In [ ]:
def inference_aware(target_loss, inference_tokens):
    '''Cheapest (N, D) reaching a target loss when inference costs 2*N FLOPs per served token.'''
    best = None
    for n in np.logspace(8, 12, 3000):
        resid = target_loss - E - A / n**alpha
        if resid <= 0:                      # this N can never reach the target, at any D
            continue
        dd = (B / resid) ** (1 / beta)      # smallest D that reaches it
        cost = 6 * n * dd + 2 * n * inference_tokens
        if best is None or cost < best[2]:
            best = (n, dd, cost)
    return best

target = 2.05
print(f"target loss {target}\n")
print(f"{'inference tokens served':>24} {'N*':>9} {'D*':>12} {'tokens/param':>14}")
for inf in [0, 1e11, 1e12, 1e13, 1e14]:
    n, dd, c = inference_aware(target, inf)
    print(f"{inf:>24.0e} {n/1e9:>7.1f}B {dd/1e9:>10.0f}B {dd/n:>14.0f}")

**This is the Lesson 3 answer, made mechanical.** Hold the target loss fixed and add inference to the objective: as the volume you expect to serve grows, the optimal model gets **smaller** and the optimal token count grows — the ratio moves from ~70 tokens/param at zero inference to thousands at high volume.

Nothing about the training-compute result changed. The *objective* changed, and that is the whole explanation for why an 8B model gets trained on trillions of tokens. (Read the ratios as relative movement, not absolute truth, given the coefficient issue above.)

---
## Part 4 — BM25 and dense retrieval fail on disjoint queries

Lesson 4's claim: hybrid retrieval wins because lexical and dense search fail on *different* queries. Demonstrate it on a 20-document support corpus with two query types — **synonym queries** with no lexical overlap, and **rare exact strings** an embedder would smooth away.

> ⚠️ **Mechanism demo, not a benchmark.** The "dense encoder" below is a hand-built concept mapping, not a trained model — it is a stand-in that makes the *failure modes* visible on a corpus small enough to read. Real recall numbers come from real embedders on your own corpus.

In [ ]:
DOCS = [
    "reset your password from the account settings page",
    "the login page rejects credentials after five failed attempts",
    "error ERR_1042 means the payment gateway timed out",
    "refunds are issued to the original payment method within ten days",
    "how to change the email address associated with your account",
    "two factor authentication can be enabled in security settings",
    "the mobile app crashes on startup after the version 4.2 update",
    "shipping delays affect orders placed during the holiday period",
    "error ERR_2071 indicates an expired session token",
    "you can cancel a subscription from the billing dashboard",
    "invoices can be downloaded as pdf from the billing dashboard",
    "the desktop client freezes when syncing large folders",
    "supported browsers and minimum operating system versions",
    "how to add a colleague to your organisation workspace",
    "api rate limits are one thousand requests per hour per key",
    "webhook deliveries retry three times before being dropped",
    "your data is exported as a zip archive within twenty four hours",
    "the search index updates within five minutes of a change",
    "single sign on is available on enterprise plans only",
    "contact support through the help widget in the bottom right",
]

QUERIES = [                                                  # (query, gold doc id)
    ("i forgot my passphrase how do i restore access", 0),   # synonym, zero lexical overlap
    ("what does ERR_1042 mean", 2),                          # rare exact string
    ("money back on a purchase", 3),                         # synonym
    ("app keeps quitting when i open it", 6),                # synonym
    ("ERR_2071", 8),                                         # rare exact string
    ("get my invoice as a pdf", 10),                         # both signals present
    ("program hangs while uploading a big directory", 11),   # synonym
    ("how many api calls am i allowed", 14),                 # partial overlap
]

STOP = set("the a an of to in on for from your you is are be can how do i my it as and with per".split())
def tok(s):
    return [w for w in s.lower().replace(",", " ").split() if w not in STOP]

In [ ]:
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [tok(d) for d in docs]; self.k1, self.b = k1, b
        self.dl = np.array([len(d) for d in self.docs], float); self.avgdl = self.dl.mean()
        self.vocab = sorted({w for d in self.docs for w in d})
        self.idx = {w: i for i, w in enumerate(self.vocab)}
        self.tf = np.zeros((len(docs), len(self.vocab)))
        for i, d in enumerate(self.docs):
            for w in d:
                self.tf[i, self.idx[w]] += 1
        df = (self.tf > 0).sum(0)
        self.idf = np.log(1 + (len(docs) - df + 0.5) / (df + 0.5))

    def scores(self, q):
        s = np.zeros(len(self.docs))
        for w in tok(q):
            if w not in self.idx:
                continue
            j = self.idx[w]; f = self.tf[:, j]
            s += self.idf[j] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.dl / self.avgdl))
        return s

# A stand-in "dense encoder": words map to concepts, documents to normalized concept vectors.
CONCEPTS = {
    "auth":     "password passphrase credentials login signin access restore reset authentication token session sign".split(),
    "account":  "account email address profile settings user colleague organisation workspace".split(),
    "payment":  "payment refund refunds money back purchase billing subscription gateway invoice invoices".split(),
    "app":      "app mobile crashes crashing quitting startup version update desktop client freezes hangs program".split(),
    "files":    "download downloaded pdf export zip archive folders directory uploading syncing large big".split(),
    "shipping": "shipping orders delivery delays holiday".split(),
    "error":    "error err timed timeout expired failed rejects".split(),
    "api":      "api rate limits requests calls key webhook deliveries allowed hour".split(),
}
CIDS = sorted(CONCEPTS)
W2C = {w: c for c, ws in CONCEPTS.items() for w in ws}

def embed(text):
    v = np.zeros(len(CIDS))
    for w in tok(text):
        c = W2C.get(w)
        if c:
            v[CIDS.index(c)] += 1
    n = np.linalg.norm(v)
    return v / n if n else v

DOC_VECS = np.stack([embed(d) for d in DOCS])
def dense_scores(q):
    return DOC_VECS @ embed(q)

In [ ]:
def ranked_list(scores, top_n=10):
    '''A real retriever returns only documents it actually matched, best first.'''
    return [i for i in np.argsort(-scores) if scores[i] > 0][:top_n]

def rrf(lists, n_docs, kappa=60):
    '''Reciprocal rank fusion: rank-based, so no score calibration between systems.'''
    s = np.zeros(n_docs)
    for lst in lists:
        for rank, doc in enumerate(lst, start=1):
            s[doc] += 1.0 / (kappa + rank)
    return s

bm = BM25(DOCS)
hits = {"BM25": 0, "dense": 0, "hybrid": 0}

print(f"{'query':<46} {'BM25':>6} {'dense':>7} {'hybrid':>8}    <- rank of the gold document")
for q, gold in QUERIES:
    lb, ld = ranked_list(bm.scores(q)), ranked_list(dense_scores(q))
    lh = ranked_list(rrf([lb, ld], len(DOCS)))
    for name, l in [("BM25", lb), ("dense", ld), ("hybrid", lh)]:
        hits[name] += int(gold in l[:3])
    fmt = lambda l: str(l.index(gold) + 1) if gold in l else "-"
    print(f"{q:<46} {fmt(lb):>6} {fmt(ld):>7} {fmt(lh):>8}")

n = len(QUERIES)
print(f"\nrecall@3:   BM25 {hits['BM25']}/{n}    dense {hits['dense']}/{n}    hybrid {hits['hybrid']}/{n}")

**The failures are disjoint, and that is the whole argument.** BM25 misses every synonym query — "passphrase" and "password" share no token — while the dense retriever misses `ERR_1042` and `ERR_2071` entirely, because a rare identifier is exactly what a fixed-width semantic vector smooths away.

Fusion recovers both: **8/8 against 5/8 and 6/8**. Note *why* RRF can do this without tuning — it consumes only ranks, so BM25 scores and cosine similarities never have to be put on a comparable scale.

Two properties of the implementation above are worth carrying into a real system: a retriever returns only documents it actually matched (an empty list is a legitimate answer), and the fusion constant $\kappa \approx 60$ damps the influence of any single system's top rank.

---
## Part 5 — Is the improvement real?

Lesson 5's central drill: 71% → 74% on 200 examples. Compute the standard error, then run the correct **paired** analysis.

In [ ]:
def se_binomial(p, n):
    return math.sqrt(p * (1 - p) / n)

p_hat, n_eval = 0.72, 200
se = se_binomial(p_hat, n_eval)
print(f"accuracy ~{p_hat:.2f} on n={n_eval}:  SE = {se:.4f}  ->  95% interval is +/- {1.96*se*100:.1f} points")
print("A 3-point gap between two such estimates is comfortably inside noise.\n")

def mcnemar_exact(b, c):
    '''Two-sided exact test on the discordant pairs only: b fixed, c broken.'''
    n = b + c
    tail = sum(comb(n, i) for i in range(min(b, c) + 1)) / 2**n
    return min(1.0, 2 * tail)

print(f"{'fixed (b)':>10} {'broken (c)':>11} {'net gain':>10} {'exact p':>9}")
for b, c in [(12, 6), (16, 10), (20, 8), (30, 12), (40, 16)]:
    print(f"{b:>10} {c:>11} {100*(b-c)/n_eval:>9.1f}% {mcnemar_exact(b, c):>9.3f}")

**The first row is the drill.** A 3-point net gain built from 12 fixes and 6 breaks gives $p \approx 0.24$ — not significant. The same 3-point gain from 16 fixes and 10 breaks is *weaker* evidence ($p \approx 0.33$), because more churn means more noise. The aggregate number cannot tell those two situations apart; only the discordant counts can.

Row 3 is the useful contrast: **6 points from 20 fixes and 8 breaks reaches $p \approx 0.04$** on the same 200 examples. So the sample size is not the whole story — effect size and churn matter just as much.

In [ ]:
def power(n, p_a, p_fix, p_break, trials=600, seed=0):
    '''How often would we detect a genuinely better system B at n examples?'''
    rng = np.random.default_rng(seed)
    sig = 0
    for _ in range(trials):
        a = rng.random(n) < p_a                            # system A correct?
        fixed  = (~a) & (rng.random(n) < p_fix)            # B fixes some of A's errors
        broken =   a  & (rng.random(n) < p_break)          # B breaks some of A's correct answers
        b_ok = (a | fixed) & ~broken
        if mcnemar_exact(int((~a & b_ok).sum()), int((a & ~b_ok).sum())) < 0.05:
            sig += 1
    return sig / trials

ns = [200, 500, 1000, 2000, 4000]
powers = [power(n, 0.71, 0.20, 0.05) for n in ns]

print("B genuinely fixes 20% of A's errors and breaks 5% of its correct answers (~+3.3 points true effect)\n")
print(f"{'n':>7} {'power':>8}")
for n, pw in zip(ns, powers):
    print(f"{n:>7} {pw:>8.2f}")

fig, ax = plt.subplots()
ax.plot(ns, powers, "o-")
ax.axhline(0.8, ls="--", c="crimson", label="conventional 80% power")
ax.set_xscale("log"); ax.set_xlabel("evaluation set size (log scale)"); ax.set_ylabel("probability of detecting a real +3-point gain")
ax.set_ylim(0, 1.02); ax.set_title("A real improvement is invisible at n = 200"); ax.legend(); plt.show()

**At n = 200 you would miss a genuinely better system roughly seven times out of eight.** Detecting a 3-point effect reliably takes a couple of thousand paired examples. That is the number to have in mind when someone proposes shipping on a 200-item eval set — and the reason the useful move is often to *read the flipped examples* rather than to run another test.

In [ ]:
rng = np.random.default_rng(1)
n_eval, p_true, n_variants, trials = 200, 0.71, 20, 3000

best = np.array([rng.binomial(n_eval, p_true, size=n_variants).max() / n_eval for _ in range(trials)])

print(f"{n_variants} IDENTICAL prompt variants, every one truly {p_true:.0%}, each scored on n={n_eval}:\n")
print(f"  mean of the BEST observed score:       {best.mean():.3f}   (+{100*(best.mean()-p_true):.1f} points of pure selection bias)")
print(f"  95th percentile of the best:           {np.quantile(best, 0.95):.3f}")
print(f"  runs where the 'winner' beats the true rate by >= 3 points: {(best >= p_true + 0.03).mean():.0%}")
print("\nNo variant is better than any other. Every point of that gap is selection on noise.")

**This is the multiple-comparisons trap, quantified.** Twenty identical variants produce an apparent winner about **6 points above the truth**, and in 99% of runs that winner "beats" the baseline by at least 3 points — the exact size of improvement teams routinely ship.

The fix is one line of process: **re-measure the selected variant on a held-out set it was never selected on**, and report that number. The same logic applies to picking a checkpoint by validation score, choosing the best of several seeds, or tuning a decoding configuration on the test set.

---
## What to carry into the interview

| Verified here | The sentence it earns you |
|---|---|
| 16 bytes/param, only 2 of them weights | "Freezing the base deletes 14 of the 16 bytes — that's what PEFT is." |
| 112 → 14.4 → 3.9 GB | "LoRA removes the optimizer term; QLoRA additionally shrinks the frozen base. Different savings." |
| Rank 12 captures ~98% at <5% of parameters | "Low-rank works because the update *is* low-rank — and fails when it isn't." |
| Nucleus 4 vs 25,000 tokens | "A fixed k is wrong for both a confident and an open context; entropy should set the cut." |
| Nucleus 638 → 33,652 across T | "Temperature is applied before truncation, so the two knobs interact." |
| Optimum is interior; ratio shifts with inference | "Compute-optimal isn't deployment-optimal — inference cost scales with parameters." |
| The Chinchilla parametric-fit discrepancy | "The 20:1 rule comes from the isoFLOP analysis; the published parametric fit doesn't reproduce it." |
| Hybrid 8/8 vs 5/8 and 6/8 | "They fail on disjoint queries, and RRF fuses them without calibrating scores." |
| McNemar 12-vs-6 → p ≈ 0.24 | "Three points on 200 examples isn't an improvement, it's noise." |
| Power 0.13 at n=200 | "Detecting a 3-point effect needs a couple of thousand paired examples." |
| +6 points from 20 identical variants | "If you picked the best of twenty, that number is inflated — re-measure on held-out data." |

**Next:** [Mock Round — The LLM Round](06_mock_round.md).